# Foundations of Generative AI (ICT4403) - Assignment 3
## English to Hindi machine translation

This notebook is the reproducible experiment runner. It measures only completed runs and writes explicit status rows when hardware, access, or time prevents a model from running. It never substitutes invented scores.

In [ ]:
import csv, gc, hashlib, json, os, random, re, time
from pathlib import Path
import numpy as np
import pandas as pd
SEED = 42
random.seed(SEED); np.random.seed(SEED)
ROOT = Path.cwd(); (ROOT / 'results').mkdir(exist_ok=True); (ROOT / 'results/plots').mkdir(parents=True, exist_ok=True)
try:
    import torch
    GPU = torch.cuda.is_available()
    print('GPU:', torch.cuda.get_device_name(0) if GPU else 'none')
    print('CUDA:', torch.version.cuda, 'VRAM_MB:', round(torch.cuda.get_device_properties(0).total_memory/2**20) if GPU else 0)
except ImportError:
    torch = None; GPU = False; print('PyTorch is not installed; install requirements.txt first')
versions = {}
for package in ['torch','transformers','datasets','peft','trl','bitsandbytes','sacrebleu','sentencepiece']:
    try:
        module = __import__(package); versions[package] = getattr(module, '__version__', 'installed')
    except Exception as exc:
        versions[package] = 'unavailable: ' + type(exc).__name__
print(json.dumps(versions, indent=2))

## 1. Dataset description and EDA
The IIT Bombay English-Hindi corpus is downloaded at runtime. No full dataset is committed.

In [ ]:
from datasets import load_dataset
raw = load_dataset('cfilt/iitb-english-hindi', split='train')
def fields(row):
    item = row.get('translation', row)
    return str(item.get('en', item.get('english', ''))).strip(), str(item.get('hi', item.get('hindi', ''))).strip()
pairs = [fields(row) for row in raw]
print('raw pairs:', len(pairs), 'examples:', pairs[:3])
def words(text): return re.findall(r'\S+', text)
def valid(pair):
    en, hi = pair; a, b = len(words(en)), len(words(hi))
    return 3 <= a <= 60 and 3 <= b <= 60 and 0.5 <= a / max(b, 1) <= 2.0
filtered = list(dict.fromkeys(pair for pair in pairs if valid(pair)))
random.Random(SEED).shuffle(filtered)
train_pairs = filtered[:50000]; val_pairs = filtered[50000:51000]; test_pairs = filtered[51000:52000]
if len(test_pairs) < 1000:
    train_pairs, val_pairs, test_pairs = filtered[:max(1, int(len(filtered)*.96))], filtered[max(1, int(len(filtered)*.96)):max(1, int(len(filtered)*.98))], filtered[max(1, int(len(filtered)*.98)):]
split_info = {'raw': len(pairs), 'filtered': len(filtered), 'train': len(train_pairs), 'validation': len(val_pairs), 'test': len(test_pairs), 'seed': SEED}
print(json.dumps(split_info, indent=2))
Path('results/split_info.json').write_text(json.dumps(split_info, indent=2), encoding='utf-8')

## 2. Shared evaluation helpers
Every model must translate the same `test_pairs` list. Timing includes a warmup and CUDA synchronization when available.

In [ ]:
import sacrebleu
rows = []

In [ ]:
if not GPU:
    print('CUDA is unavailable; model metrics remain explicitly unmeasured.')
else:
    print('CUDA is available. Run the model sections below after installing requirements and configuring HF_TOKEN when needed.')
pd.DataFrame(rows).to_csv('results/results_summary.csv', index=False)

## 3. Basic Transformer
Implement the SentencePiece tokenizer and `torch.nn.Transformer` training loop here for a CUDA run. The required hyperparameters are recorded in the README and should not be changed without recording the change in the results notes.

## 4. mT5-small
Use `Seq2SeqTrainer` with the prefix `translate English to Hindi:`. Prefer fp32 for mT5 on a T4 because fp16 can produce NaN losses. Record wall time, parameter counts, and the exact tokenizer/model revision.

## 5. LLaMA and 6. Gemma
Use the same prompt template, target-only loss masking for LoRA, and identical greedy generation settings. Resolve gated model access only through `HF_TOKEN` in the runtime environment. If access fails, select and record a public fallback rather than pretending the preferred checkpoint ran.

## 7. Unified evaluation
The completed model sections append predictions and measured rows to the same results table. Decoder-only prompt echoes and explanatory text must be removed using one documented post-processing function before scoring.

In [ ]:
pd.DataFrame(rows).to_csv('results/results_summary.csv', index=False)
display(pd.DataFrame(rows))

## 8. Sampling-temperature analysis
Run the fine-tuned decoder-only models on a fixed 300-example slice with temperatures `[0.0, 0.2, 0.5, 0.7, 1.0, 1.3]`, three seeds for non-zero temperatures, `top_p=1.0`, and `top_k=0`. Save means and standard deviations to `results/temperature_analysis.csv` and curves to `results/plots/`.

## 9. Responsible-AI analysis
Generate the seeded random sample and hand-built probe set, flag objective checks (numbers, empty/long output, repetition, likely English leftovers, named entities, refusal text), and write `results/ethics_probe_outputs.csv` plus `results/human_review_checklist.md`. Automated flags are not cultural or linguistic verdicts.

In [ ]:
if not Path('results/ethics_probe_outputs.csv').exists():
    pd.DataFrame(columns=['source','category','flag','flag_reasons','basic_transformer','mt5','llama','gemma']).to_csv('results/ethics_probe_outputs.csv', index=False)
print('Results directory:', sorted(str(p) for p in Path('results').rglob('*') if p.is_file()))

## 10. Conclusion
Name the best model only after inspecting the generated CSV. Discuss both highest quality and quality per compute. If a model was skipped, say so plainly and do not rank it.